### Building a RAG System with Chroma and langchain

### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)



In [3]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma

from typing import List
import numpy as np

In [4]:
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



### Sample Data

In [5]:
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs

['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [7]:
import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i+1}.txt", 'w') as f:
        f.write(doc)

print(f"Sample docs created!! in {temp_dir}")        

Sample docs created!! in /var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5


In [9]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

dir_load = DirectoryLoader(temp_dir, glob="*.txt", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
documents = dir_load.load()

print(f"Documents Loaded: {len(documents)}")
print(documents[0].page_content[:100])

Documents Loaded: 3

    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction bet


### Document Splitter


In [15]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, length_function=len, separators=["\n\n", "\n", " ", ""])
chunks = text_splitter.split_documents(documents)

print(f"Length od chunks, {len(chunks)}")
for idx, chunk in enumerate(chunks):
    print(f"Chunk {idx+1}: {chunk.page_content[:100]}")
    print(f"Metadata {idx+1}: {chunk.metadata}")

Length od chunks, 7
Chunk 1: Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between 
Metadata 1: {'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_3.txt'}
Chunk 2: Deep Learning and Neural Networks
Metadata 2: {'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_2.txt'}
Chunk 3: Deep learning is a subset of machine learning based on artificial neural networks. 
    These networ
Metadata 3: {'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_2.txt'}
Chunk 4: excel at sequential data processing.
Metadata 4: {'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_2.txt'}
Chunk 5: Machine Learning Fundamentals
Metadata 5: {'source': '/var/folders/nv/3g6f6py91psc7jgvpcl6t6980000gn/T/tmp0sqw4ly5/doc_1.txt'}
Chunk 6: Machine learning is a subset of artificial intelligence that enables systems to learn 
    and impro
Metadata 6: {'source': '/var/folders/

### Embedding Models


In [ ]:
os.environ["OPEN_API_KEY"] = os.getenv("OPEN_API_KEY")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 
vector = embeddings.embed_query("sample text")
vector


### Initialize chroma DB vector and store vector embeddings in it